# Bayesian Spatial Modeling of Business Success

This notebook implements a Bayesian Hierarchical model to analyze the success rate of businesses across municipalities in São Paulo (SP) state, incorporating spatial random effects (ICAR).

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
from shapely import wkt
import matplotlib.pyplot as plt
import seaborn as sns

# Spatial analysis
import libpysal as lps
from esda.moran import Moran
from spreg import lm
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Bayesian modeling
import pymc as pm
import pytensor.tensor as pt
import scipy.stats as stats

import os
from pathlib import Path

import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid", palette="viridis")

## 1. Data Loading and Filtering

We load the enhanced dataset and filter for the state of São Paulo (SP).

In [ ]:
DATA_PATH = Path("../../data/processed/final_integrated_dataset_enhanced.csv")

df_raw = pd.read_csv(DATA_PATH)

# Filter for São Paulo (SP abbreviation in this dataset)
df_sp = df_raw[df_raw['state'] == 'SP'].copy()

# Convert to GeoDataFrame
df_sp['geometry'] = df_sp['geometry'].apply(wkt.loads)
gdf_sp = gpd.GeoDataFrame(df_sp, geometry='geometry', crs="EPSG:4326")

print(f"Total municipalities in SP: {len(gdf_sp)}")
gdf_sp.head()

## 2. Variable Standardization

Standardizing variables using z-score ensures that model coefficients are comparable and improves MCMC convergence.

In [ ]:
# Define predictors based on user request and dataset columns
predictors = [
    'HDI_income', 
    'road_density_km_km2', 
    'ndbi', 
    'GDP_per_capita'
]

# Target variable: Success Rate (Active / Total)
# We use 'active' and 'total_count' (which is active + failed)
gdf_sp['total_count'] = gdf_sp['active'] + gdf_sp['failed']
gdf_sp['success_rate'] = gdf_sp['active'] / gdf_sp['total_count']

# Standardize (Z-score)
for col in predictors:
    if col in gdf_sp.columns:
        gdf_sp[f'z_{col}'] = (gdf_sp[col] - gdf_sp[col].mean()) / gdf_sp[col].std()

## 3. Spatial Weight Matrix

Defining the neighborhood structure using the Queen adjacency criterion.

In [ ]:
w = lps.weights.Queen.from_dataframe(gdf_sp)
w.transform = 'r'
adj_matrix = w.sparse

print(f"Number of islands: {len(w.islands)}")

## 4. Diagnostics

### 4.1. Spatial Autocorrelation (Global Moran's I)
Checking if successful cities cluster together.

In [ ]:
mi = Moran(gdf_sp['success_rate'], w)
print(f"Moran's I: {mi.I:.4f}")
print(f"P-value: {mi.p_sim:.4f}")

# Visualization
fig, ax = plt.subplots(figsize=(10, 8))
gdf_sp.plot(column='success_rate', legend=True, ax=ax, cmap='RdYlGn')
ax.set_title("Success Rate by Municipality (SP)")
plt.axis('off')
plt.show()

### 4.2. Multicollinearity (VIF)
Ensuring predictors are not too redundant.

In [ ]:
z_predictors = [f'z_{col}' for col in predictors if f'z_{col}' in gdf_sp.columns]
X_vif = gdf_sp[z_predictors].fillna(0)
vif_data = pd.DataFrame()
vif_data["feature"] = X_vif.columns
vif_data["VIF"] = [variance_inflation_factor(X_vif.values, i) for i in range(len(X_vif.columns))]

print(vif_data.sort_values('VIF', ascending=False))

### 4.3. Spatial Heterogeneity
Comparing success rates across different internal regions (e.g., mesoregions). 

In [ ]:
# Note: If 'mesoregion' column is not available, you may need to join it from IBGE metadata
if 'mesoregion' in gdf_sp.columns:
    plt.figure(figsize=(12, 6))
    sns.boxplot(data=gdf_sp, x='mesoregion', y='success_rate')
    plt.xticks(rotation=45)
    plt.title("Success Rate Distribution by Mesoregion")
    plt.show()

    # ANOVA test
    groups = [group['success_rate'].values for name, group in gdf_sp.groupby('mesoregion')]
    f_stat, p_val = stats.f_oneway(*groups)
    print(f"ANOVA F-statistic: {f_stat:.4f}, P-value: {p_val:.4f}")
else:
    print("⚠️ Mesoregion column not found. Heterogeneity analysis skipped.")

### 4.4. Lagrange Multiplier (LM) Tests
Diagnostic for spatial lag vs spatial error using OLS residuals.

In [ ]:
from spreg import OLS

y_lm = gdf_sp['success_rate'].values.reshape(-1, 1)
X_lm = X_vif.values

ols_model = OLS(y_lm, X_lm, w=w, spat_diag=True, name_y='success_rate', name_x=z_predictors)
print(ols_model.summary)

## 5. Bayesian Spatial Model (PyMC Implementation)

We implement a Bayesian Binomial model with an ICAR spatial random effect.

In [ ]:
def get_adj_structure(w):
    """Converts libpysal weights to adjacency structure for ICAR"""
    adj = w.neighbors
    # Map IDs to integers 0..N-1
    id_map = {id: i for i, id in enumerate(w.id_order)}
    
    node1 = []
    node2 = []
    for node, neighbors in adj.items():
        for neighbor in neighbors:
            node1.append(id_map[node])
            node2.append(id_map[neighbor])
                
    return np.array(node1), np.array(node2)

def ICAR(name, w_obj, node_size, tau):
    """
    Intrinsic Conditional Autoregressive (ICAR) prior implementation.
    """
    node1, node2 = get_adj_structure(w_obj)
    
    def logp(phi):
        # The ICAR log-likelihood is proportional to:
        # -0.5 * tau * sum_{i < j} w_ij * (phi_i - phi_j)^2
        phi_diff = phi[node1] - phi[node2]
        return -0.5 * tau * pt.sum(phi_diff**2)
    
    phi_raw = pm.Normal(f"{name}_raw", mu=0, sigma=1, shape=node_size)
    # Centering constraint for identifiability
    phi = pm.Deterministic(name, phi_raw - pt.mean(phi_raw))
    pm.Potential(f"{name}_potential", logp(phi))
    
    return phi

def run_sp_success_model(data, w_obj):
    # Standardized Predictors (X)
    z_cols = [f'z_{c}' for c in ['HDI_income', 'road_density_km_km2', 'ndbi', 'GDP_per_capita']]
    X = data[z_cols].fillna(0).values
    
    # Counts
    successes = data['active'].values
    total_trials = (data['active'] + data['failed']).values
    n_municipalities = len(data)

    with pm.Model() as spatial_model:
        # 1. Priors for Global Intercept and Coefficients
        alpha = pm.Normal("alpha", mu=0, sigma=3)
        beta = pm.Normal("beta", mu=0, sigma=1, shape=X.shape[1])
        
        # 2. ICAR Spatial Random Effect (phi)
        # tau_phi controls the strength of spatial smoothing
        sigma_phi = pm.Exponential("sigma_phi", lam=1.0)
        tau_phi = 1.0 / (sigma_phi**2)
        
        phi = ICAR("phi", w_obj, n_municipalities, tau_phi)
        
        # 3. Link Function (Logit)
        mu = alpha + pt.dot(X, beta) + phi
        p = pm.Deterministic("p_success", pm.math.invlogit(mu))
        
        # 4. Binomial Likelihood
        y = pm.Binomial("y_obs", n=total_trials, p=p, observed=successes)
        
        # 5. Sampling (Using 1000 for local pilot)
        trace = pm.sample(1000, tune=1000, target_accept=0.9, chains=2)
        
    return trace

### 6. Model Execution

Run the model on the São Paulo data.

In [ ]:
# To run the model, uncomment the lines below:
# trace = run_sp_success_model(gdf_sp, w)
# pm.summary(trace, var_names=['alpha', 'beta', 'sigma_phi'])